# Giai đoạn 2 — Mục 2.1, 2.3, 2.4 — Huấn luyện và đánh giá CNN raw và envelope với LOLO
**Đầu ra**:
- `outputs/tables/cnn_raw_lolo_results.csv`
- `outputs/tables/cnn_env_lolo_results.csv`
- Các model `.h5` và `.tflite` trong `outputs/models/`

## Phạm vi

**MLP là pipeline chính.** CNN1D raw/envelope chỉ được chạy như **thí nghiệm mở rộng** để so sánh tài nguyên (số tham số, kích thước `.tflite`, kết quả INT8, lịch sử huấn luyện). CNN không phải RQ chính.

Đối với CNN-envelope, chuẩn hóa dùng `StandardScaler` fit trên toàn bộ giá trị của **Train trong từng LOLO fold**, sau đó transform Val/Test bằng cùng scaler. Không dùng scaler toàn cục của toàn dataset.


In [30]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle
from sklearn.metrics import f1_score
from tensorflow.keras.callbacks import EarlyStopping

from common import models, quantization, training

In [31]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

windows_raw_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/windows_cnn_raw.parquet")
windows_env_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/windows_cnn_env.parquet")

In [32]:
manifest_filtered = pd.read_csv("../giai_doan_1_tien_xu_ly/outputs/tables/manifest_filtered.csv")

# Hàm trích xuất dữ liệu cho CNN

In [33]:
def get_cnn_data_unscaled(df, window_col='window'):
    """Chỉ trích xuất mảng numpy, chưa chuẩn hóa"""
    X = np.stack(df[window_col].values).astype(np.float32)
    y = df['label'].values
    return X, y

# 1DCNN (raw/env) - LOLO

In [34]:
def get_cnn_data_unscaled(df, window_col='window'):
    """Chỉ trích xuất mảng numpy, chưa chuẩn hóa"""
    X = np.stack(df[window_col].values).astype(np.float32)
    y = df['label'].values
    return X, y

def run_cnn_lolo(df, window_col, build_model_func, window_size, model_name, standardize_envelope=False):
    results = []
    
    # Tạo thư mục lưu lịch sử huấn luyện
    HISTORY_DIR = OUTPUT_DIR / "history"
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    
    for fold_info, train_df, val_df, test_df in training.iterate_lolo_splits(df, load_col='load_hp'):
        print(f"\n--- {model_name} - Fold: {fold_info['fold_name']} ---")
        
        # 1. Lấy dữ liệu CHƯA chuẩn hóa
        X_train_raw, y_train = get_cnn_data_unscaled(train_df, window_col)
        X_val_raw, y_val = get_cnn_data_unscaled(val_df, window_col)
        X_test_raw, y_test = get_cnn_data_unscaled(test_df, window_col)
        
        # 2. Xử lý chuẩn hóa theo tham số standardize_envelope
        if standardize_envelope:
            # Min-Max Scaling (theo window) cho Envelope
            X_train_min = X_train_raw.min(axis=1, keepdims=True)
            X_train_max = X_train_raw.max(axis=1, keepdims=True)
            X_train = (X_train_raw - X_train_min) / (X_train_max - X_train_min + 1e-8)
            
            X_val_min = X_val_raw.min(axis=1, keepdims=True)
            X_val_max = X_val_raw.max(axis=1, keepdims=True)
            X_val = (X_val_raw - X_val_min) / (X_val_max - X_val_min + 1e-8)
            
            X_test_min = X_test_raw.min(axis=1, keepdims=True)
            X_test_max = X_test_raw.max(axis=1, keepdims=True)
            X_test = (X_test_raw - X_test_min) / (X_test_max - X_test_min + 1e-8)
        else:
            # StandardScaler toàn cục (trên tập Train) cho Raw
            global_mean = X_train_raw.mean()
            global_std = X_train_raw.std() + 1e-8
            
            X_train = (X_train_raw - global_mean) / global_std
            X_val = (X_val_raw - global_mean) / global_std
            X_test = (X_test_raw - global_mean) / global_std
        
        # Thêm chiều channel cho CNN (shape: N, Window, 1)
        X_train = X_train[..., np.newaxis]
        X_val = X_val[..., np.newaxis]
        X_test = X_test[..., np.newaxis]
        
        # Mã hóa nhãn
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_val_enc = le.transform(y_val)
        y_test_enc = le.transform(y_test)
        
        # 3. Khởi tạo và Huấn luyện mô hình
        model = build_model_func(window_size=window_size)
        model = models.compile_classifier(model)
        
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        history = model.fit(X_train, y_train_enc,
                            validation_data=(X_val, y_val_enc),
                            epochs=100, batch_size=64,
                            callbacks=[early_stop], verbose=0)
        
        # 4. LƯU LỊCH SỬ HUẤN LUYỆN
        history_df = pd.DataFrame(history.history)
        history_csv_path = HISTORY_DIR / f"{model_name}_{fold_info['fold_name']}_history.csv"
        history_df.to_csv(history_csv_path, index=False)
        
        # 5. Đánh giá Float Model
        loss, acc = model.evaluate(X_test, y_test_enc, verbose=0)
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        f1 = f1_score(y_test_enc, y_pred, average='macro')
        
        # 6. Lượng tử hóa và Đánh giá INT8
        tflite_bytes = quantization.quantize_model_int8(model, X_train)
        int8_result = quantization.evaluate_tflite_model(tflite_bytes, X_test, y_test_enc)
        
        param_count = int(model.count_params())
        tflite_size_bytes = int(len(tflite_bytes))
        tflite_size_kb = tflite_size_bytes / 1024.0
        
        results.append({
            'fold': fold_info['fold_name'],
            'test_load': fold_info['test_load'],
            'float_accuracy': acc,
            'float_f1': f1,
            'int8_accuracy': int8_result['accuracy'],
            'epochs': len(history_df),
            'num_parameters': param_count,
            'tflite_size_bytes': tflite_size_bytes,
            'tflite_size_kb': tflite_size_kb
        })
        
        # Lưu Models
        model.save(MODELS_DIR / f"{model_name}_{fold_info['fold_name']}.h5")
        tflite_path = MODELS_DIR / f"{model_name}_{fold_info['fold_name']}.tflite"
        quantization.model_bytes_to_file(tflite_bytes, tflite_path)
    
    return pd.DataFrame(results)

In [35]:
print("Các cột của manifest_filtered:", manifest_filtered.columns)
print("Index của manifest_filtered:", manifest_filtered.index.name)
display(manifest_filtered.head())

Các cột của manifest_filtered: Index(['file_path', 'load_hp', 'label', 'fault_diameter_mils', 'or_position',
       'source_category', 'sensor_location', 'declared_sample_rate_khz',
       'n_samples_DE', 'n_samples_FE', 'n_samples_BA', 'rpm_from_file',
       'read_error', 'warnings', 'has_warning'],
      dtype='str')
Index của manifest_filtered: None


,file_path,load_hp,label,fault_diameter_mils,or_position,source_category,sensor_location,declared_sample_rate_khz,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning
0,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,122571,122571.0,122571.0,1796.0,NaN,NaN,False
1,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,1,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121410,121410.0,121410.0,1772.0,NaN,NaN,False
2,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,2,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1748.0,NaN,NaN,False
3,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,3,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1722.0,NaN,NaN,False
4,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,14.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121846,121846.0,121846.0,1796.0,NaN,NaN,False


In [36]:
# 1. Khôi phục lại cột file_id cho manifest_filtered (Khớp định dạng với file 04 Giai đoạn 1)
manifest_filtered['file_id'] = manifest_filtered.apply(
    lambda row: f"{row['label']}_{row['load_hp']}_{row.get('fault_diameter_mils', '')}_{row['file_path']}", axis=1
)

# 2. Lấy metadata và thực hiện Merge
metadata = manifest_filtered[['file_id', 'load_hp']]

windows_raw_df = windows_raw_df.merge(metadata, on='file_id', how='left')
windows_env_df = windows_env_df.merge(metadata, on='file_id', how='left')

# Loại bỏ các dòng NaN (nếu có do merge không khớp)
windows_raw_df = windows_raw_df.dropna(subset=['load_hp']).reset_index(drop=True)
windows_env_df = windows_env_df.dropna(subset=['load_hp']).reset_index(drop=True)

# 3. Ép kiểu load_hp về int để hàm training.iterate_lolo_splits hoạt động đúng
windows_raw_df['load_hp'] = windows_raw_df['load_hp'].astype(int)
windows_env_df['load_hp'] = windows_env_df['load_hp'].astype(int)

print("Các cột của windows_raw_df sau khi merge:", windows_raw_df.columns)

Các cột của windows_raw_df sau khi merge: Index(['file_id', 'label', 'start_idx', 'window', 'load_hp'], dtype='str')


In [37]:
cnn_raw_results = run_cnn_lolo(
    windows_raw_df, 'window',
    models.build_cnn1d_raw, 2048, 'cnn_raw',
    standardize_envelope=False
)
cnn_raw_results.to_csv(TABLES_DIR / "cnn_raw_lolo_results.csv", index=False)


--- cnn_raw - Fold: test_load_0 ---


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpnt0u6tdx\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpnt0u6tdx\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpnt0u6tdx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2048, 1), dtype=tf.float32, name='raw_signal')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2058240292432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058310785104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058310783952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058240293776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058310794320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058240293008: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


...

f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- cnn_raw - Fold: test_load_3 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpgmb7w8ua\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpgmb7w8ua\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpgmb7w8ua'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2048, 1), dtype=tf.float32, name='raw_signal')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2058247603792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058247610128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058247610320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058247615120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058247607824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058247615696: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [38]:
cnn_env_results = run_cnn_lolo(
    windows_env_df, 'window',
    models.build_cnn1d_env, 1024, 'cnn_env',
    standardize_envelope=True
)
cnn_env_results.to_csv(TABLES_DIR / "cnn_env_lolo_results.csv", index=False)


--- cnn_env - Fold: test_load_0 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp_3pbz62f\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp_3pbz62f\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmp_3pbz62f'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1024, 1), dtype=tf.float32, name='envelope')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2058252567376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252569488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252574864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252574672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252569872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252570064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252569680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252582160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252566992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252576784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058252576400

f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- cnn_env - Fold: test_load_1 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpdgb68pqj\assets


...

f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- cnn_env - Fold: test_load_3 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp48nvbsn2\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp48nvbsn2\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmp48nvbsn2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1024, 1), dtype=tf.float32, name='envelope')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2058258164496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258161232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258167760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258168336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258168528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258169488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258162960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258164304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258162000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258163152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2058258165264

f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [39]:
cnn_raw_results.head()

,fold,test_load,float_accuracy,float_f1,int8_accuracy,epochs,num_parameters,tflite_size_bytes,tflite_size_kb
0,test_load_0,0,0.900386,0.906222,0.899614,20,2652,8888,8.679688
1,test_load_1,1,0.985611,0.984439,0.975801,12,2652,8888,8.679688
2,test_load_2,2,0.981675,0.980082,0.990838,16,2652,8888,8.679688
3,test_load_3,3,0.962842,0.959483,0.944589,13,2652,8888,8.679688


In [40]:
cnn_env_results.head()

,fold,test_load,float_accuracy,float_f1,int8_accuracy,epochs,num_parameters,tflite_size_bytes,tflite_size_kb
0,test_load_0,0,0.771209,0.773215,0.763916,35,1444,8536,8.335938
1,test_load_1,1,0.823357,0.814265,0.815224,23,1444,8536,8.335938
2,test_load_2,2,0.838216,0.833097,0.836589,20,1444,8536,8.335938
3,test_load_3,3,0.736125,0.708389,0.738397,27,1444,8536,8.335938


In [41]:
# Tổng hợp tài nguyên MLP vs CNN phục vụ Bảng 2b/Bảng bổ sung
mlp_size_path = Path('../giai_doan_2_xay_dung_mo_hinh/outputs/tables/mlp_model_size.csv')
local_mlp_size = TABLES_DIR / 'mlp_model_size.csv'

size_frames = []
if local_mlp_size.exists():
    size_frames.append(pd.read_csv(local_mlp_size))
else:
    print(f'Chưa tìm thấy {local_mlp_size}; hãy chạy 02_mlp_lolo_revised.ipynb trước.')

for name in ['cnn_raw', 'cnn_env']:
    p = TABLES_DIR / f'{name}_model_size.csv'
    if p.exists():
        size_frames.append(pd.read_csv(p))

if size_frames:
    model_size_comparison = pd.concat(size_frames, ignore_index=True)
    model_size_comparison.to_csv(TABLES_DIR / 'model_size_comparison_mlp_cnn.csv', index=False)
    display(model_size_comparison)


Chưa tìm thấy outputs\tables\mlp_model_size.csv; hãy chạy 02_mlp_lolo_revised.ipynb trước.
